# SenioCare — Ollama Model Server (Colab)

This notebook runs **Ollama only** as a remote model server.
The SenioCare FastAPI app runs on Replit/Render and connects to this Colab via ngrok.

**Architecture:**
```
Flutter → Replit (FastAPI + ADK + Tools + DB) → Colab (Ollama GPU)
```

> Run cells 1-5 in order. Cell 5 gives you the ngrok URL to paste in Replit.

## Step 1 — Install Ollama

In [ ]:
# Install Ollama binary
!curl -fsSL https://ollama.com/install.sh | sh
print("Ollama installed")

## Step 2 — Start Ollama Server

In [ ]:
import subprocess, time, requests

# Start Ollama as a background process
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Wait for it to be ready
OLLAMA_BASE_URL = "http://localhost:11434"
for i in range(30):
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=2)
        if r.status_code == 200:
            print(f"Ollama server ready (attempt {i+1})")
            break
    except:
        time.sleep(1)
else:
    print("WARNING: Ollama may not have started")

## Step 3 — Pull the Model

In [ ]:
# Configure model
MODEL_NAME = "gemma4:e4b"

def model_exists(name):
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
        return any(name in m for m in models)
    except:
        return False

if model_exists(MODEL_NAME):
    print(f"Model '{MODEL_NAME}' already available")
else:
    print(f"Pulling '{MODEL_NAME}' — this may take a while...")
    !ollama pull {MODEL_NAME}
    print(f"Model '{MODEL_NAME}' ready")

!ollama list

## Step 4 — Verify Model Works

In [ ]:
# Quick test — make sure the model responds
import json

test_payload = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Say hello in Arabic"}],
    "stream": False
}

r = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=test_payload, timeout=120)
if r.status_code == 200:
    msg = r.json().get("message", {}).get("content", "")
    print(f"Model test OK: {msg[:200]}")
else:
    print(f"Model test FAILED: {r.status_code} {r.text[:200]}")

## Step 5 — Expose via ngrok

This creates a public URL that the deployed SenioCare app connects to.

**Copy the ngrok URL and paste it as `OLLAMA_BASE_URL` in your Replit/Render environment.**

In [ ]:
!pip -q install pyngrok
from pyngrok import ngrok, conf

# Paste your ngrok auth token here
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"

if NGROK_AUTH_TOKEN != "YOUR_NGROK_AUTH_TOKEN_HERE":
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    ngrok.kill()  # kill any existing tunnels
    tunnel = ngrok.connect(11434, "http")
    PUBLIC_URL = tunnel.public_url
    print("="*60)
    print(f"  OLLAMA PUBLIC URL: {PUBLIC_URL}")
    print("="*60)
    print()
    print("Copy this URL and set it as OLLAMA_BASE_URL:")
    print()
    print(f"  In .env file:")
    print(f"    OLLAMA_BASE_URL={PUBLIC_URL}")
    print()
    print(f"  Or in Replit/Render environment variables:")
    print(f"    OLLAMA_BASE_URL = {PUBLIC_URL}")
    print()
    print("The SenioCare app will route all LLM calls to this Colab.")
    print("Tools, callbacks, DB — everything else runs on Replit.")
else:
    print("Set NGROK_AUTH_TOKEN above, then re-run this cell.")
    print(f"Local URL (Colab only): {OLLAMA_BASE_URL}")

## Step 6 — Test from External (Optional)

Run this after Step 5 to verify the ngrok tunnel works.

In [ ]:
# Test the public URL
if 'PUBLIC_URL' in globals():
    test_payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Say hello"}],
        "stream": False
    }
    r = requests.post(f"{PUBLIC_URL}/api/chat", json=test_payload, timeout=120)
    print(f"Public URL test: {r.status_code}")
    if r.status_code == 200:
        print(f"Response: {r.json().get('message', {}).get('content', '')[:200]}")
else:
    print("Run Step 5 first to get the public URL")

## Keep Alive

Run this cell to keep the Colab runtime alive while the model server is running.
The runtime will stay active as long as this cell is executing.

In [ ]:
import time
print("Keeping Colab alive... (Ctrl+C or stop button to end)")
print(f"Model: {MODEL_NAME}")
if 'PUBLIC_URL' in globals():
    print(f"Public URL: {PUBLIC_URL}")
print()

try:
    while True:
        # Health check every 60s
        try:
            r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
            models = len(r.json().get("models", []))
            print(f"[{time.strftime('%H:%M:%S')}] Ollama alive — {models} models loaded", end="\r")
        except:
            print(f"[{time.strftime('%H:%M:%S')}] WARNING: Ollama not responding", end="\r")
        time.sleep(60)
except KeyboardInterrupt:
    print("\nStopped.")

## Shutdown (Optional)

In [ ]:
if 'ollama_proc' in globals() and ollama_proc is not None:
    ollama_proc.terminate()
    print("Ollama stopped")

try:
    ngrok.kill()
    print("ngrok tunnels closed")
except:
    pass